<a href="https://colab.research.google.com/github/general-molecular-simulations/so3lr/blob/main/examples/so3lr_colab_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Install SO3LR
#@markdown ## Select Runtime → Change runtime type → Hardware accelerator → T4 GPU (or another available GPU), save, then run this cell.
#@markdown SO3LR supports Python 3.12 and 3.13. Installation may take several minutes.
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
if sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError(f"SO3LR needs Python 3.12 or 3.13; this runtime has {sys.version.split()[0]}.")

_setup_flag = "/content/.SO3LR_SETUP"
has_gpu = subprocess.run("nvidia-smi -L", shell=True, stdout=subprocess.DEVNULL,
                         stderr=subprocess.DEVNULL).returncode == 0

def sh(cmd):
    print(f"$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout, end="", flush=True)
    return result.returncode

if not os.path.exists(_setup_flag):
    # Colab's internal constraints can conflict with the required package versions.
    for _var in ("PIP_CONSTRAINT", "UV_CONSTRAINT"):
        os.environ.pop(_var, None)
    use_uv = sh("pip install -q uv==0.11.26") == 0

    def install(spec):
        if use_uv and sh(f"uv pip install --system -q {spec}") == 0:
            return
        if sh(f"pip install -q {spec}") != 0:
            raise RuntimeError(f"Installation failed: {spec}")

    if not has_gpu:
        print("No GPU detected: using the slower CPU build. Select Runtime -> Change runtime type -> T4 GPU for GPU acceleration.")
    install("ase py3Dmol")
    if not os.path.exists("/content/so3lr"):
        if sh("git clone -q https://github.com/general-molecular-simulations/so3lr.git /content/so3lr") != 0:
            raise RuntimeError("SO3LR clone failed; rerun this cell.")
    # Install JAX and its matching CUDA plugin together. Require the tested Flax version.
    install(('"/content/so3lr[cuda12]"' if has_gpu else '/content/so3lr')
            + ' "flax>=0.12.8,<0.12.9"')

# Colab may retain newer CUDA 13 plugins after JAX is downgraded. The cuda12 extra
# supplies our matching backend. Also repair sessions with an existing setup marker.
if has_gpu:
    _cuda13_packages = []
    for _package in ("jax-cuda13-plugin", "jax-cuda13-pjrt"):
        try:
            version(_package)
        except PackageNotFoundError:
            continue
        _cuda13_packages.append(_package)
    if _cuda13_packages:
        print("Removing incompatible preinstalled CUDA 13 plugins:", ", ".join(_cuda13_packages))
        result = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", *_cuda13_packages], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(result.stdout, end="")
        result.check_returncode()

# Import JAX-MD only in a child process so GPU allocations are released afterwards.
_verification_code = """
import sys
import jax
import jax_md
import so3lr
backend = jax.default_backend()
if sys.argv[1] == "gpu" and backend != "gpu":
    raise RuntimeError("A GPU is attached but JAX is using the CPU. Restart the session and rerun Install SO3LR.")
print(f"SO3LR {so3lr.__version__} ready. JAX {jax.__version__}, backend: {backend}")
"""
try:
    result = subprocess.run([sys.executable, "-c", _verification_code, "gpu" if has_gpu else "cpu"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout, end="")
    result.check_returncode()
except subprocess.CalledProcessError as exc:
    if os.path.exists(_setup_flag):
        os.remove(_setup_flag)
    raise RuntimeError("SO3LR verification failed; see the error above. Use Runtime -> Restart session, then rerun Install SO3LR.") from exc

# Never mark an unsuccessful installation complete.
with open(_setup_flag, "w"):
    pass


## Running simulations

SO3LR ships a command-line interface; this notebook wraps it in interactive forms.

1. Select a GPU runtime and run **Install SO3LR** once in each new runtime.
2. Use **Run an SO3LR simulation** to optimize a geometry (`opt`), run molecular
   dynamics (`nvt`, `npt`, `nve`), or evaluate energies/forces on a dataset (`eval`).
3. Use **Visualize a trajectory** to animate the result.

The default MD run is **0.5 ps** (50 cycles × 20 steps × 0.5 fs). MD first relaxes
the input geometry; its relaxation frames are saved separately as
`<output-stem>_opt.xyz`. The first run also compiles the model with JAX, which can
take several minutes before simulation progress appears.

Change any field and rerun the cell as needed. Runs replace their selected output
and associated relaxation file; choose a different output name to keep a result.
Input structures are protected from these output paths.

The bundled example `/content/so3lr/examples/data/ala15_folded.xyz` is a folded Ala15
peptide **in vacuum**. The default `lr_cutoff = 100` Å covers this peptide. For larger
gas-phase systems, choose a cutoff larger than the maximum interatomic separation,
including room for motion and the 2 Å dispersion switching region. For a periodic
box, 12 Å is a starting value; check convergence for your system. NPT requires a
fully periodic structure with a valid cell.

To use your own system, upload an ASE-readable structure through the **Files**
panel (the folder icon on the left), then enter its path. To save results locally,
refresh the Files panel, open a result's menu, and select **Download**. Files under
`/content` are temporary and can be lost when Colab deletes or replaces the runtime;
download results you want to keep.


In [ ]:
#@title Run an SO3LR simulation
#@markdown Pick a simulation type, point to an input structure, set the parameters, and run.

simulation_type = "nvt" #@param ["opt", "nvt", "npt", "nve", "eval"]
input_file = "/content/so3lr/examples/data/ala15_folded.xyz" #@param {type:"string"}
output_file = "" #@param {type:"string"}
model = "so3lr-tauri-m-2606" #@param ["so3lr-tauri-m-2606", "so3lr-tauri-s-2606", "so3lr-tauri-l-2606", "so3lr"]

#@markdown **Common options** — `lr_cutoff` in Å: 100 covers the bundled vacuum peptide; choose a larger value for larger gas-phase systems. `total_charge` is in units of e; evaluation can instead use charges from the input file.
lr_cutoff = 100.0 #@param {type:"number"}
total_charge = 0 #@param {type:"integer"}

#@markdown **Molecular dynamics only** (`nvt` / `npt` / `nve`) — temperature in K, timestep in fs
temperature = 300.0 #@param {type:"number"}
timestep_fs = 0.5 #@param {type:"number"}
md_cycles = 50 #@param {type:"integer"}
md_steps = 20 #@param {type:"integer"}
seed = 0 #@param {type:"integer"}
pressure_bar = 1.0 #@param {type:"number"}

#@markdown **Optimization only** (`opt`) — force convergence in eV/Å
force_conv = 0.05 #@param {type:"number"}

#@markdown **Evaluation only** (`eval`) — comma-separated targets, e.g. `forces,dipole_vec`. Error metrics need reference values in the input (e.g. forces in an extxyz file); without them, as for the bundled peptide, they print as `nan` and only the predictions are saved.
eval_targets = "forces" #@param {type:"string"}
#@markdown Keep `eval_use_file_charges` enabled to use each structure's `charge` metadata (0 if absent). Disable it to apply `total_charge` to every structure, including a neutral override of 0.
eval_use_file_charges = True #@param {type:"boolean"}

import os
import shlex
import subprocess
from pathlib import Path

# NPT needs full 3D periodicity; the bundled peptide is in vacuum. Say so plainly
# rather than letting the run fail with a traceback halfway through.
if simulation_type == "npt":
    import ase.io
    input_pbc = ase.io.read(input_file).get_pbc()
    assert input_pbc.all(), (
        f"{input_file} must be periodic in all three axes for NPT; "
        f"received pbc={input_pbc.tolist()}. Use nvt / nve / opt, or "
        "supply a fully periodic structure with a cell.")

if not output_file:
    stem = os.path.splitext(os.path.basename(input_file))[0]
    output_file = f"{stem}_{simulation_type}.xyz"

# Repeated runs may replace outputs, but must never replace the input structure.
input_path = Path(input_file).expanduser().resolve()
output_path = Path(output_file).expanduser().resolve()
protected_outputs = [output_path]
if simulation_type in ("nvt", "npt", "nve"):
    protected_outputs.append(output_path.with_name(f"{output_path.stem}_opt.xyz"))
for candidate in protected_outputs:
    if candidate.resolve() == input_path:
        raise ValueError(
            f"Output path {candidate} would overwrite the input structure. "
            "Choose a different output_file."
        )

common = ["--model", model, "--lr-cutoff", str(lr_cutoff),
          # Without this, re-running the cell aborts because the output exists.
          "--overwrite"]
# Omitting --total-charge preserves per-structure charge metadata for eval.
if simulation_type != "eval" or not eval_use_file_charges:
    common += ["--total-charge", str(total_charge)]

if simulation_type == "eval":
    cmd = ["so3lr", "eval", "--datafile", input_file, "--save-to", output_file] + common
    if eval_targets.strip():
        cmd += ["--targets", eval_targets.strip()]
else:
    cmd = ["so3lr", simulation_type, "--input", input_file, "--output", output_file] + common
    if simulation_type == "opt":
        cmd += ["--force-conv", str(force_conv)]
    else:  # nvt / npt / nve
        cmd += ["--temperature", str(temperature), "--dt", str(timestep_fs),
                "--md-cycles", str(md_cycles), "--md-steps", str(md_steps),
                "--seed", str(seed)]
        if simulation_type == "npt":
            cmd += ["--pressure", str(pressure_bar)]

print("Running:\n  " + " ".join(shlex.quote(c) for c in cmd) + "\n" + "-" * 70)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
except BaseException:
    # Stopping this cell must also stop the simulation and release its GPU memory.
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
    raise
finally:
    proc.stdout.close()
print("-" * 70)
if code == 0:
    last_output = output_file  # picked up by the visualization cell
    print(f"Done. Output written to: {output_file}")
else:
    raise subprocess.CalledProcessError(code, cmd)


In [ ]:
#@title Visualize a trajectory
#@markdown Leave blank to use the output of the cell above. Multi-frame files are animated.

trajectory_file = "" #@param {type:"string"}

import os
from io import StringIO

import ase.io
import py3Dmol

trajectory_file = trajectory_file.strip() or globals().get("last_output", "")
assert trajectory_file, "Run a simulation first, or type a path above."
assert os.path.exists(trajectory_file), f"File not found: {trajectory_file}"

frames = list(ase.io.read(trajectory_file, index=":"))
print(f"Loaded {len(frames)} frame(s) from {trajectory_file}")

buf = StringIO()
for atoms in frames:
    ase.io.write(buf, atoms, format="xyz")

view = py3Dmol.view(width=800, height=500)
# Store each XYZ structure as an animation frame (the 3Dmol trajectory API).
view.addModelsAsFrames(buf.getvalue(), "xyz")
view.setStyle({"sphere": {"radius": 0.4}, "stick": {"radius": 0.2}})
view.setBackgroundColor("white")
view.zoomTo()
if len(frames) > 1:
    view.animate({"loop": "forward", "interval": 100})
view.show()
